## Tools

In [ ]:
from utils.files import load_json

tools_path = r"C:\Users\david\projects\ba\src\data\tools.json"
tools = load_json(tools_path)

for t in tools["tools"]:
    print(t)

{'name': 'get_energy_consumption', 'description': 'Liefert den Energieverbrauch eines Haushalts für einen bestimmten Zeitraum.', 'parameters': {'type': 'object', 'properties': {'household_id': {'type': 'string', 'description': 'Eindeutige ID des Haushalts.'}, 'start_date': {'type': 'string', 'description': 'Startdatum im Format YYYY-MM-DD.'}, 'end_date': {'type': 'string', 'description': 'Enddatum im Format YYYY-MM-DD.'}}, 'required': ['household_id', 'start_date', 'end_date']}}
{'name': 'get_weather', 'description': 'Liefert Wetterdaten für einen bestimmten Ort und Zeitraum.', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'Ort, für den die Wetterdaten abgerufen werden sollen.'}, 'date': {'type': 'string', 'description': 'Datum im Format YYYY-MM-DD.'}}, 'required': ['location', 'date']}}
{'name': 'calculate_average', 'description': 'Berechnet den Durchschnitt einer Liste numerischer Werte.', 'parameters': {'type': 'object', 'properties': 

## Systemprompt

In [11]:
import json

system_prompt = f"""
  Du bist ein Tool-Orchestrator.

  Erzeuge für die Benutzeranfrage einen ausführbaren Toolplan.

  Verfügbare Tools:
  {json.dumps(tools, indent=2)}

  Der Output MUSS exakt diesem Schema entsprechen:

  {{
    "plan": [
      {{
        "id": "step_1",
        "tool": "tool_name",
        "parameters": {{}}
      }}
    ]
  }}

  Regeln:

  1. Verwende ausschließlich die angegebenen Tools.
  2. Jeder Schritt benötigt eine eindeutige ID.
  3. Die Parameter müssen zum jeweiligen Tool passen.
  4. Ein späterer Schritt darf auf das Ergebnis eines vorherigen
    Schrittes verweisen.
  5. Verwende dafür:
    {{
      "$ref": "step_X.result.path"
    }}
  6. Gib ausschließlich valides JSON zurück.
  7. Führe die Tools nicht selbst aus.
"""

In [13]:
user_prompt = """
Ermittle den durchschnittlichen Energieverbrauch des Haushalts
HH001 im Januar 2026.
"""

In [15]:
import requests

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:8b",
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "stream": False
    }
)

result = response.json()

print(result["message"]["content"])

{
  "plan": [
    {
      "id": "step_1",
      "tool": "get_energy_consumption",
      "parameters": {
        "household_id": "HH001",
        "start_date": "2026-01-01",
        "end_date": "2026-01-31"
      }
    },
    {
      "id": "step_2",
      "tool": "calculate_average",
      "parameters": {
        "values": [
          {
            "$ref": "step_1.result.values"
          }
        ]
      }
    }
  ]
}


In [10]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

model = "qwen3:8b"

def get_local_response(
    user_prompt: str,
    system_prompt: str = system_prompt,
    model: str = model
) -> dict:
    return client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

NameError: name 'system_prompt' is not defined